In [1]:
import plotly.io as pio
# pio.renderers.default = "browser"
import os

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd


In [2]:
blendshape_names = ['_neutral', 'browDownLeft', 'browDownRight', 'browInnerUp',
       'browOuterUpLeft', 'browOuterUpRight', 'cheekPuff', 'cheekSquintLeft',
       'cheekSquintRight', 'eyeBlinkLeft', 'eyeBlinkRight', 'eyeLookDownLeft',
       'eyeLookDownRight', 'eyeLookInLeft', 'eyeLookInRight', 'eyeLookOutLeft',
       'eyeLookOutRight', 'eyeLookUpLeft', 'eyeLookUpRight', 'eyeSquintLeft',
       'eyeSquintRight', 'eyeWideLeft', 'eyeWideRight', 'jawForward',
       'jawLeft', 'jawOpen', 'jawRight', 'mouthClose', 'mouthDimpleLeft',
       'mouthDimpleRight', 'mouthFrownLeft', 'mouthFrownRight', 'mouthFunnel',
       'mouthLeft', 'mouthLowerDownLeft', 'mouthLowerDownRight',
       'mouthPressLeft', 'mouthPressRight', 'mouthPucker', 'mouthRight',
       'mouthRollLower', 'mouthRollUpper', 'mouthShrugLower',
       'mouthShrugUpper', 'mouthSmileLeft', 'mouthSmileRight',
       'mouthStretchLeft', 'mouthStretchRight', 'mouthUpperUpLeft',
       'mouthUpperUpRight', 'noseSneerLeft', 'noseSneerRight'],

In [3]:
def plot_blendshape_changes(df, blendshape_names):
    """
    Plots the changes of blendshape scores over frames using Plotly.
    
    Args:
        df (pd.DataFrame): A DataFrame where rows represent frames and columns represent blendshape names.
    """
    fig = go.Figure()
    
    # Assuming the first column is the frame index
    frames = df.index
    
    for blendshape in df.columns:
        if blendshape not in blendshape_names:
            continue
        fig.add_trace(go.Scatter(x=frames, y=df[blendshape], mode='lines', name=blendshape))
    
    fig.update_layout(
        title='Blendshape Changes Over Frames',
        xaxis_title='Frame',
        yaxis_title='Blendshape Score',
        legend_title='Blendshapes',
        yaxis=dict(range=[0, 1]),
        height=1000,
        width=1000
    )
    
    return fig.show()

In [4]:
# blendshape_paths = ["/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/"+fn for fn in os.listdir("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/") if fn.endswith('.csv')]
# df = pd.read_csv(blendshape_paths[0])
# plot_blendshape_changes(df, ['eyeBlinkLeft', 'eyeBlinkRight'])
# analyze_slices(df, slice_size=18000, step=100, blink_threshold=0.5, yawn_threshold=0.5, nap_threshold=30, merge_gap=5, filename="").iloc[:,264:]

In [12]:
def analyze_slices(df, slice_size=1800, step=100, blink_threshold=0.5, yawn_threshold=0.5, nap_threshold=30, merge_gap=5, filename=""):
    """
    Analyzes dataset slices to compute summary statistics and detect blinks and yawns.
    
    Args:
        df (pd.DataFrame): The dataset with blendshape scores.
        slice_size (int): Number of rows per slice.
        step (int): Number of frames the slice moves forward at each step.
        blink_threshold (float): Threshold for detecting blinks.
        yawn_threshold (float): Threshold for detecting yawns.
        merge_gap (int): Frames within which consecutive blinks/yawns are merged.
        filename (str): Name of the file the data comes from.
    
    Returns:
        pd.DataFrame: A DataFrame containing summary statistics for each slice.
    """
    results = []
    num_slices = int((len(df) - slice_size) // step + 1) # Adjust to calculate the number of slices based on step
    
    for i in range(num_slices):
        start_frame = int(i * step)
        end_frame = int(start_frame + slice_size - 1)
        
        slice_df = df.iloc[start_frame:end_frame + 1]
        summary = {
            'slice_index': i,
            'filename': filename,
            'start_frame': start_frame,
            'end_frame': end_frame
        }
        
        # Compute summary statistics
        for stat in ['mean', 'median', 'std', 'min', 'max']:
            summary.update({f'{col}_{stat}': getattr(slice_df[col], stat)() for col in slice_df.columns})
        
        def detect_events(series, threshold):
            above_thresh = series > threshold
            events = []
            start = None
            
            for idx, val in enumerate(above_thresh):
                if val:
                    if start is None:
                        start = idx
                else:
                    if start is not None:
                        events.append((start, idx - 1))
                        start = None
            if start is not None:
                events.append((start, len(series) - 1))
            
            merged_events = []
            for event in events:
                if merged_events and event[0] - merged_events[-1][1] <= merge_gap:
                    merged_events[-1] = (merged_events[-1][0], event[1])
                else:
                    merged_events.append(event)
            
            durations = [e[1] - e[0] + 1 for e in merged_events]
            
            return durations
        
            return {
                'count': len(durations),
                'mean_duration': pd.Series(durations).mean() if durations else 0,
                'median_duration' : pd.Series(durations).median() if durations else 0,
                'std_duration': pd.Series(durations).std() if len(durations) > 1 else 0,
                'min_duration': min(durations) if durations else 0,
                'max_duration': max(durations) if durations else 0,
            }
        
        yawn_durations  = detect_events(slice_df['jawOpen'], 
                                   yawn_threshold)
        
        blink_durations = detect_events(slice_df[['eyeBlinkLeft', 'eyeBlinkRight']].max(axis=1), 
                                    blink_threshold)
        nap_durations   = [blink_duration for blink_duration in blink_durations if blink_duration> nap_threshold]
        blink_durations = [blink_duration for blink_duration in blink_durations if blink_duration<=nap_threshold]
        
        
        event_names = ['yawn', 'blink', 'nap']
        for event_name in event_names:
            durations = eval(event_name + '_durations')
            
            summary[f'{event_name}s_count']           = len(durations)
            summary[f'{event_name}s_mean_duration']   = pd.Series(durations).mean() if durations else 0
            summary[f'{event_name}s_median_duration'] = pd.Series(durations).median() if durations else 0
            summary[f'{event_name}s_std_duration']    = pd.Series(durations).std() if len(durations) > 1 else 0
            summary[f'{event_name}s_min_duration']    = min(durations) if durations else 0
            summary[f'{event_name}s_max_duration']    = max(durations) if durations else 0
        
        results.append(summary)
    
    return pd.DataFrame(results)

def analyze_directory(directory, slice_size=1800, step=100, blink_threshold=0.5, yawn_threshold=0.5, nap_threshold=30, merge_gap=5):
    """
    Processes all CSV files in a directory, analyzes them, and concatenates results.
    
    Args:
        directory (str): Path to the directory containing CSV files.
        slice_size (int): Number of rows per slice.
        blink_threshold (float): Threshold for detecting blinks.
        yawn_threshold (float): Threshold for detecting yawns.
        merge_gap (int): Frames within which consecutive blinks/yawns are merged.
    
    Returns:
        pd.DataFrame: A DataFrame with statistics for all slices of all videos.
    """
    all_results = []
    
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            print(filename)
            filepath = os.path.join(directory, filename)
            try:
                df = pd.read_csv(filepath)
            except:
                print(f"problem reading file {filepath}")
                continue
            result_df = analyze_slices(df, slice_size, step, blink_threshold, yawn_threshold, nap_threshold, merge_gap, filename)
            all_results.append(result_df)
    
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


In [21]:
slice_size, step, blink_yawn_threshold, nap_threshold, merge_gap = '100_100_0.7_45_3'.split('_')
slice_size = int(slice_size)
step = int(step)
blink_yawn_threshold = float(blink_yawn_threshold)
nap_threshold = int(nap_threshold)
merge_gap = int(merge_gap)

# slice_size = 1800
# step = 100
# blink_yawn_threshold = 0.4
# nap_threshold = 45
# merge_gap = 3

r = analyze_directory(directory="/work/long_lab/Mojtaba/data/DROZY/DROZY/videos_30fps/blendshapes/", 
                      slice_size=slice_size, 
                      step=step, 
                      blink_threshold=blink_yawn_threshold, 
                      yawn_threshold=blink_yawn_threshold, 
                      nap_threshold=nap_threshold,
                      merge_gap=merge_gap)

path_to_save = '/work/long_lab/Mojtaba/data/DROZY/DROZY/videos_30fps/blendshapes/Summary/'
r.to_csv(f"{path_to_save}Blendshapes_Summary_{slice_size}_{step}_{blink_yawn_threshold}_{nap_threshold}_{merge_gap}.csv")

0_68_6_30fps_blendshape.csv
0_72_2_30fps_blendshape.csv
0_74_7_30fps_blendshape.csv
0_73_6_30fps_blendshape.csv
0_74_8_30fps_blendshape.csv
0_67_9_30fps_blendshape.csv
0_73_3_30fps_blendshape.csv
0_68_8_30fps_blendshape.csv
0_70_7_30fps_blendshape.csv
0_74_5_30fps_blendshape.csv
0_69_6_30fps_blendshape.csv
0_64_8_30fps_blendshape.csv
0_69_8_30fps_blendshape.csv
0_63_3_30fps_blendshape.csv
0_61_7_30fps_blendshape.csv
0_63_4_30fps_blendshape.csv
0_62_7_30fps_blendshape.csv
0_61_6_30fps_blendshape.csv
0_62_3_30fps_blendshape.csv
0_62_6_30fps_blendshape.csv
0_63_2_30fps_blendshape.csv
0_61_3_30fps_blendshape.csv
0_64_4_30fps_blendshape.csv
0_65_7_30fps_blendshape.csv
0_65_3_30fps_blendshape.csv
0_65_8_30fps_blendshape.csv
0_64_9_30fps_blendshape.csv
0_66_2_30fps_blendshape.csv
0_66_7_30fps_blendshape.csv
0_66_3_30fps_blendshape.csv
0_67_4_30fps_blendshape.csv
0_68_2_30fps_blendshape.csv
0_71_7_30fps_blendshape.csv
0_71_4_30fps_blendshape.csv
0_70_3_30fps_blendshape.csv


In [15]:
# slice_size = 50
# step = 50
# blink_yawn_threshold = 0.4
# nap_threshold = 45
# merge_gap = 3
# r = analyze_directory(directory="/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/", 
#                       slice_size=slice_size, 
#                       step=step, 
#                       blink_threshold=blink_yawn_threshold, 
#                       yawn_threshold=blink_yawn_threshold, 
#                       nap_threshold=nap_threshold,
#                       merge_gap=merge_gap)
# path_to_save = '/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/'
# r.to_csv(f"{path_to_save}Blendshapes_Summary_{slice_size}_{step}_{blink_yawn_threshold}_{nap_threshold}_{merge_gap}.csv")

2_21_10_30fps_blendshape.csv
1_04_0_30fps_blendshape.csv
1_07_10_30fps_blendshape.csv
1_07_0_30fps_blendshape.csv
1_01_0_30fps_blendshape.csv
1_05_10_30fps_blendshape.csv
2_20_5_30fps_blendshape.csv
4_39_5_30fps_blendshape.csv
4_39_10_30fps_blendshape.csv
5_50_10_30fps_blendshape.csv
5_59_0_30fps_blendshape.csv
5_59_5_30fps_blendshape.csv
5_59_10_30fps_blendshape.csv
5_57_5_30fps_blendshape.csv
4_45_5_30fps_blendshape.csv
problem reading file /home/mojtaba.kananisarche/Zh/UTA/Blendshapes/4_45_5_30fps_blendshape.csv
1_06_5_30fps_blendshape.csv
1_06_10_30fps_blendshape.csv
1_12_10_30fps_blendshape.csv
1_08_0_30fps_blendshape.csv
1_12_0_30fps_blendshape.csv
1_04_10_30fps_blendshape.csv
1_04_5_30fps_blendshape.csv
1_03_10_30fps_blendshape.csv
1_11_10_30fps_blendshape.csv
1_03_5_30fps_blendshape.csv
3_29_0_30fps_blendshape.csv
1_12_5_30fps_blendshape.csv
1_11_0_30fps_blendshape.csv
3_29_10_30fps_blendshape.csv
1_11_5_30fps_blendshape.csv
1_07_5_30fps_blendshape.csv
1_08_10_30fps_blendshape.

In [130]:
# r.to_csv("/home/mojtaba.kananisarche/Zh/UTA/Blendshapes/Summary/Blendshapes_Summary_nap_1800_100.csv")